In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SpambaseClassification") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/10 20:25:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
names_file = "/Users/tcbailey/Documents/imse785/projects/project5/inputs/spambase/spambase.names"

column_names = []

with open(names_file, "r") as f:
    for line in f:
        line = line.strip()

        if (
            ":" in line
            and line.startswith((
                "word_freq_",
                "char_freq_",
                "capital_run_length"
            ))
        ):
            column_names.append(line.split(":")[0])

# Add the target column
column_names.append("spam")

print(f"Number of columns found: {len(column_names)}")

Number of columns found: 58


In [3]:
# Load spambase.data (no header, comma-separated values) into a DataFrame
spambase_df = spark.read.csv("/Users/tcbailey/Documents/imse785/projects/project5/inputs/spambase/spambase.data", \
    header=False, inferSchema=True).toDF(*column_names)

print("Rows, Columns:", spambase_df.count(), len(spambase_df.columns))
spambase_df.show(5)

Rows, Columns: 4601 58


26/07/10 20:25:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+-----------------+-------------+------------+-------------+--------------+----------------+------------------+---------------+--------------+-----------------+--------------+----------------+----------------+-------------------+--------------+------------------+---------------+-------------+----------------+--------------+--------------+-------------+---------------+------------+-------------+----------------+-------------+-------------+--------------+----------------+-------------+--------------+-------------+------------+--------------------+--------------+---------------+------------+----------------+------------+-----------------+------------------+-----------------+------------+-------------+---------------+--------------------+-----------+-----------+-----------+-----------+-----------+-----------+--------------------------+--------------------------+------------------------+----+
|word_freq_make|word_freq_address|word_freq_all|word_freq_3d|word_freq_our|word_freq

In [4]:
# Q1 - Train/Test Split, Best Model per Algorithm (LR, DT, RF)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

feature_cols = [c for c in spambase_df.columns if c != "spam"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Train/Test Split (70/30)
train_df, test_df = spambase_df.randomSplit([0.7, 0.3], seed=42)
train_df.cache()
test_df.cache()

DataFrame[word_freq_make: double, word_freq_address: double, word_freq_all: double, word_freq_3d: double, word_freq_our: double, word_freq_over: double, word_freq_remove: double, word_freq_internet: double, word_freq_order: double, word_freq_mail: double, word_freq_receive: double, word_freq_will: double, word_freq_people: double, word_freq_report: double, word_freq_addresses: double, word_freq_free: double, word_freq_business: double, word_freq_email: double, word_freq_you: double, word_freq_credit: double, word_freq_your: double, word_freq_font: double, word_freq_000: double, word_freq_money: double, word_freq_hp: double, word_freq_hpl: double, word_freq_george: double, word_freq_650: double, word_freq_lab: double, word_freq_labs: double, word_freq_telnet: double, word_freq_857: double, word_freq_data: double, word_freq_415: double, word_freq_85: double, word_freq_technology: double, word_freq_1999: double, word_freq_parts: double, word_freq_pm: double, word_freq_direct: double, word

In [5]:
# Evaluation Helper (Precison, Recall, F1)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(predictions, label_col="spam", prediction_col="prediction"):
    evaluator_f1 = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol=prediction_col, metricName="f1")
    evaluation_precision = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol=prediction_col, metricName="weightedPrecision")
    evaluation_recall = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol=prediction_col, metricName="weightedRecall")

    f1 = evaluator_f1.evaluate(predictions)
    precision = evaluation_precision.evaluate(predictions)
    recall = evaluation_recall.evaluate(predictions)

    return f1, precision, recall

In [6]:
# Logistic Regression - Grid Search (F1 criterion)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

lr = LogisticRegression(labelCol="spam", featuresCol="features", maxIter=100)
lr_pipeline = Pipeline(stages=[assembler, lr])

lr_param_grid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 1.0])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) # 0=L2, 1=L1
    .build())

lr_evaluator = MulticlassClassificationEvaluator(
    labelCol="spam", predictionCol="prediction", metricName="f1")

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps = lr_param_grid,
    evaluator=lr_evaluator,
    numFolds=5,
    parallelism=4
)

lr_cv_model = lr_cv.fit(train_df)
best_lr_model = lr_cv_model.bestModel

26/07/10 20:25:26 WARN BlockManager: Block rdd_31_0 already exists on this machine; not re-adding it
26/07/10 20:25:26 WARN BlockManager: Block rdd_31_0 already exists on this machine; not re-adding it
26/07/10 20:25:27 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [7]:
# Decision Tree - Grid Search (F1 criterion)
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(labelCol="spam", featuresCol="features", seed=42)
dt_pipeline = Pipeline(stages=[assembler, dt])

dt_param_grid = (ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10, 20])
    .addGrid(dt.minInstancesPerNode, [1, 5, 10])
    .build())

dt_evaluator = lr_evaluator  # Reuse the same evaluator for F1

dt_cv = CrossValidator(
    estimator=dt_pipeline,
    estimatorParamMaps = dt_param_grid,
    evaluator=dt_evaluator,
    numFolds=5,
    parallelism=4
)

dt_cv_model = dt_cv.fit(train_df)
best_dt_model = dt_cv_model.bestModel

In [10]:
# Random Forest - Grid Search (F1 criterion)
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="spam", featuresCol="features", seed=42)
rf_pipeline = Pipeline(stages=[assembler, rf])

rf_param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 10, 20])
    .build())

rf_evaluator = lr_evaluator  # Reuse the same evaluator for F1

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=rf_evaluator,
    numFolds=5,
    parallelism=4
)

rf_cv_model = rf_cv.fit(train_df)
best_rf_model = rf_cv_model.bestModel

26/07/10 20:30:22 WARN DAGScheduler: Broadcasting large task binary with size 1102.9 KiB
26/07/10 20:30:22 WARN DAGScheduler: Broadcasting large task binary with size 1102.9 KiB
26/07/10 20:30:23 WARN DAGScheduler: Broadcasting large task binary with size 1316.1 KiB
26/07/10 20:30:23 WARN DAGScheduler: Broadcasting large task binary with size 1316.1 KiB
26/07/10 20:30:23 WARN DAGScheduler: Broadcasting large task binary with size 1523.9 KiB
26/07/10 20:30:23 WARN DAGScheduler: Broadcasting large task binary with size 1722.7 KiB
26/07/10 20:30:23 WARN DAGScheduler: Broadcasting large task binary with size 1903.8 KiB
26/07/10 20:30:24 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/07/10 20:30:24 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/07/10 20:30:24 WARN DAGScheduler: Broadcasting large task binary with size 1196.7 KiB
26/07/10 20:30:24 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/07/10 20:30:24 WARN DAGSche

In [11]:
# Evaluate Best Models (Test and All Data)

models_q1 = {
    "Logistic Regression": best_lr_model,
    "Decision Tree": best_dt_model,
    "Random Forest": best_rf_model
}

q1_results_test = []
q1_results_all = []

for name, model in models_q1.items():
    # Test set
    test_pred = model.transform(test_df)
    p, r, f1 = evaluate_model(test_pred)
    print(f"{name} on TEST:")
    print("Precision:", p, "Recall:", r, "F1:", f1)
    q1_results_test.append((name, p, r, f1))

    # All data
    all_pred = model.transform(spambase_df)
    p_all, r_all, f1_all = evaluate_model(all_pred)
    print(f"{name} on ALL DATA:")
    print("Precision:", p_all, "Recall:", r_all, "F1:", f1_all)
    q1_results_all.append((name, p_all, r_all, f1_all))

print("Q1 Test Metrics:", q1_results_test)
print(q1_results_test[0:5])
print("Q1 All Data Metrics:", q1_results_all)
print(q1_results_all[0:5])

Logistic Regression on TEST:
Precision: 0.9152732454087354 Recall: 0.915332780728136 F1: 0.9154607768469154
Logistic Regression on ALL DATA:
Precision: 0.9179846695221805 Recall: 0.9186918063983301 F1: 0.9184959791349707
Decision Tree on TEST:
Precision: 0.9215207026506884 Recall: 0.9214976015852252 F1: 0.9215536938309216
Decision Tree on ALL DATA:
Precision: 0.9480113722179346 Recall: 0.9479950205225625 F1: 0.9480547707020213


26/07/10 20:44:34 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/07/10 20:44:34 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/07/10 20:44:34 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


Random Forest on TEST:
Precision: 0.9518811030492489 Recall: 0.9521584638586726 F1: 0.952018278750952


26/07/10 20:44:34 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/07/10 20:44:35 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/07/10 20:44:35 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


Random Forest on ALL DATA:
Precision: 0.9812707291349572 Recall: 0.9813893054944283 F1: 0.9813084112149533
Q1 Test Metrics: [('Logistic Regression', 0.9152732454087354, 0.915332780728136, 0.9154607768469154), ('Decision Tree', 0.9215207026506884, 0.9214976015852252, 0.9215536938309216), ('Random Forest', 0.9518811030492489, 0.9521584638586726, 0.952018278750952)]
[('Logistic Regression', 0.9152732454087354, 0.915332780728136, 0.9154607768469154), ('Decision Tree', 0.9215207026506884, 0.9214976015852252, 0.9215536938309216), ('Random Forest', 0.9518811030492489, 0.9521584638586726, 0.952018278750952)]
Q1 All Data Metrics: [('Logistic Regression', 0.9179846695221805, 0.9186918063983301, 0.9184959791349707), ('Decision Tree', 0.9480113722179346, 0.9479950205225625, 0.9480547707020213), ('Random Forest', 0.9812707291349572, 0.9813893054944283, 0.9813084112149533)]
[('Logistic Regression', 0.9179846695221805, 0.9186918063983301, 0.9184959791349707), ('Decision Tree', 0.9480113722179346, 0.9

In [12]:
# Q2 - 5-fold CV with ROC AUC (Best Overall Model)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

bin_evaluator = BinaryClassificationEvaluator(
    labelCol="spam",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

In [13]:
# Logistic Regression - Grid Search (ROC AUC criterion)
lr = LogisticRegression(labelCol="spam", featuresCol="features", maxIter=100)
lr_pipeline_roc = Pipeline(stages=[assembler, lr])

lr_param_grid_roc = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 1.0])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build())

lr_cv_roc = CrossValidator(
    estimator=lr_pipeline_roc,
    estimatorParamMaps=lr_param_grid_roc,
    evaluator=bin_evaluator,
    numFolds=5,
    parallelism=4
)

lr_cv_model_roc = lr_cv_roc.fit(train_df)
best_lr_model_roc = lr_cv_model_roc.bestModel
lr_best_roc = max(lr_cv_model_roc.avgMetrics)
print("LR best ROC AUC (CV):", lr_best_roc)

LR best ROC AUC (CV): 0.9645738916294608
